# IMPORT

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# DEFINE FOLDERS

In [10]:
STATION_FEATURE_FOLDER = Path("station_features")
DATASET_INPUT_FOLDER = Path("correlated_data_csv")

folders = [DATASET_INPUT_FOLDER, STATION_FEATURE_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [11]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/final_clean_data.csv")

print("(Rows, Columns):", df.shape)
print("Stations:", df["station_code"].nunique())

(Rows, Columns): (6813164, 17)
Stations: 1438


# NORMALIZE USAGE

In [12]:
df["usage_ratio"] = df["total_bikes_now"] / df["capacity"]
df["usage_ratio"] = df["usage_ratio"].clip(0, 1)

# DEFINE FEATURE EXTRACTION PER STATION

In [13]:
def extract_station_features(group):
    features = {}

    # --- Basic usage ---
    features["mean_usage"] = group["usage_ratio"].mean()
    features["std_usage"] = group["usage_ratio"].std()
    features["min_usage"] = group["usage_ratio"].min()
    features["max_usage"] = group["usage_ratio"].max()

    # --- Temporal behavior ---
    features["mean_weekend"] = group[group["is_weekend"] == 1]["usage_ratio"].mean()
    features["mean_weekday"] = group[group["is_weekend"] == 0]["usage_ratio"].mean()

    features["day_usage"] = group[group["is_day"] == 1]["usage_ratio"].mean()
    features["night_usage"] = group[group["is_day"] == 0]["usage_ratio"].mean()

    # --- Daily amplitude ---
    hourly_mean = group.groupby("hour")["usage_ratio"].mean()
    features["daily_amplitude"] = hourly_mean.max() - hourly_mean.min()

    # --- Weather sensitivity (CORRELATIONS) ---
    for col in [
        "temperature",
        "wind_speed",
        "wind_direction",
        "weather_code"
    ]:
        if group[col].nunique() > 1:
            features[f"corr_{col}"] = group["usage_ratio"].corr(group[col])
        else:
            features[f"corr_{col}"] = 0.0

    # --- Spatial info (kept for later analysis) ---
    features["latitude"] = group["latitude"].iloc[0]
    features["longitude"] = group["longitude"].iloc[0]
    features["capacity"] = group["capacity"].iloc[0]

    return pd.Series(features)


# BUILD STATION FEATURE TABLE

In [14]:
station_features = (
    df.groupby("station_code", sort=False)
    .apply(extract_station_features)
    .reset_index()
)

print("Station feature table shape:", station_features.shape)
station_features.head()

c:\Users\emirh\miniconda3\envs\dalas\Lib\site-packages\numpy\lib\_function_base_impl.py:2908: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
c:\Users\emirh\miniconda3\envs\dalas\Lib\site-packages\numpy\lib\_function_base_impl.py:2913: RuntimeWarning: invalid value encountered in dot
  c = dot(X, X_T.conj())
c:\Users\emirh\miniconda3\envs\dalas\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\emirh\miniconda3\envs\dalas\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Station feature table shape: (1438, 17)


C:\Users\emirh\AppData\Local\Temp\ipykernel_242068\991587447.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(extract_station_features)


,station_code,mean_usage,std_usage,min_usage,max_usage,mean_weekend,mean_weekday,day_usage,night_usage,daily_amplitude,corr_temperature,corr_wind_speed,corr_wind_direction,corr_weather_code,latitude,longitude,capacity
0,1001,0.559825,0.281199,0.0,1.000000,0.513568,0.578142,0.654058,0.505467,0.379505,0.018252,0.191286,NaN,-0.066237,48.857059,2.341798,17.0
1,1002,0.519805,0.300845,0.0,1.000000,0.448492,0.548044,0.614558,0.464249,0.558496,0.130809,0.177713,NaN,-0.036869,48.857993,2.346917,22.0
2,1006,0.465188,0.228937,0.0,0.939394,0.457781,0.468122,0.427181,0.485789,0.397882,0.050005,0.164658,NaN,-0.084435,48.862630,2.349807,33.0
3,1007,0.583057,0.268926,0.0,1.000000,0.537458,0.601125,0.712259,0.505538,0.529762,0.262977,0.279316,NaN,-0.032126,48.863589,2.347779,16.0
4,1008,0.504736,0.300589,0.0,1.000000,0.478562,0.515101,0.627728,0.432777,0.564778,0.225117,0.249421,NaN,-0.045336,48.863627,2.342624,32.0


# CLEAN FEATURE TABLE

In [15]:
station_features = station_features.replace([np.inf, -np.inf], np.nan)
station_features = station_features.dropna()

print("After cleaning:", station_features.shape)

After cleaning: (0, 17)


# SAVE STATION FEATURES

In [16]:
station_features.to_csv(f"{STATION_FEATURE_FOLDER}/station_features.csv", index=False)
print("Saved station_features.csv")

Saved station_features.csv
